# Ch 9. 왜 RAG가 필요한가

이 노트북은 [AI Assistant Engineering - Part 3, Ch 9](https://desty.github.io/study-ai-assistant-engineering/part3/09-why-rag/) 의 실습용 보조 자료입니다.

## 주제
- LLM의 학습 지식 한계 (지식 컷오프, 프라이빗 데이터, 최신성)
- RAG (Retrieval-Augmented Generation) 개념과 필요 이유
- LLM 단독 vs LLM + RAG 직접 비교
- 파인튜닝 vs RAG 선택 기준
- RAG 도입 전 3가지 함정

In [ ]:
!pip install -q anthropic

In [ ]:
import os
from getpass import getpass

for k in ['ANTHROPIC_API_KEY']:
    if not os.getenv(k):
        os.environ[k] = getpass(f'{k}: ')

## 1. 개념 — LLM 은 학습한 것만 안다

Claude · GPT 모델은 학습 시점까지의 데이터로만 만들어집니다.

| 모르는 것 | 왜 |
|---|---|
| **컷오프 이후 정보** | 학습 데이터에 없음 (예: 오늘 주가) |
| **프라이빗 데이터** | 공개 인터넷에 없는 것 (우리 회사 문서·DB) |
| **자주 바뀌는 데이터** | 모델 재학습 주기 >> 데이터 변경 주기 (재고·이벤트·정책) |

## 2. RAG 는 한 문장

**RAG** = 질문과 관련된 **문서를 먼저 찾아서**, 그 문서를 **프롬프트에 넣어** 답하게 하는 기법.

**RAG 의 3가지 이득**:
1. **최신성** — 문서 업데이트만 하면 모델 재학습 없이 최신 정보 반영
2. **프라이빗 지식 활용** — 회사 정책·매뉴얼·코드베이스 기반 답변
3. **추적 가능성 (citation)** — 답변이 어느 문서의 어느 부분을 근거로 했는지 표시 가능

## 4. 최소 예제 — 같은 질문, 두 경로 직접 비교

In [ ]:
from anthropic import Anthropic
client = Anthropic()

question = "우리 회사의 연차 신청 절차는?"

# 1) LLM 단독 — 모델은 우리 회사를 모름
r1 = client.messages.create(
    model="claude-haiku-4-5", max_tokens=256,
    messages=[{"role": "user", "content": question}],
)
print("=== LLM 단독 ===")
print(r1.content[0].text)
print()

In [ ]:
# 2) LLM + 수동 RAG — 우리 회사 문서를 프롬프트에 직접 주입 (단순 버전)
COMPANY_DOC = """## 연차 신청 절차 (2026 개정)
1. 사내 포털 → 휴가 > 연차 신청
2. 최소 2주 전 신청, 팀장 승인 필요
3. 연속 5일 이상은 임원 결재 추가
4. 긴급 사유는 이메일로 사전 통보 후 사후 신청 가능"""

r2 = client.messages.create(
    model="claude-haiku-4-5", max_tokens=256,
    system=f"아래 회사 문서를 바탕으로만 답하세요. 문서에 없으면 '문서에 없습니다' 라고 답하세요.\n\n{COMPANY_DOC}",
    messages=[{"role": "user", "content": question}],
)
print("=== LLM + RAG(수동) ===")
print(r2.content[0].text)

## 5. 실전 튜토리얼

### 5.1 지식 컷오프 실험

In [ ]:
questions = [
    "어제 나온 뉴스를 요약해줘",                        # 최신성
    "우리 회사의 개인정보 보호 정책 3조는?",            # 프라이빗
    "2026년 4월 현재 서울의 미세먼지는?",               # 실시간
    "파이썬 list 의 append 메서드 사용법은?",          # 학습된 공개 지식
]

for q in questions:
    r = client.messages.create(
        model="claude-haiku-4-5", max_tokens=200,
        messages=[{"role": "user", "content": q}],
    )
    print(f"\nQ: {q}")
    print(f"A: {r.content[0].text[:200]}...")

## 6. 자주 깨지는 포인트

### 실수 1. 'RAG 면 hallucination 0' 미신
문서에 정답이 있어도 모델은 **엉뚱한 재해석** 또는 **검색 결과 + 자기 추측 섞기** 가능.
RAG 는 **발생률을 낮출 뿐** 완전 방지 불가.

**대응**: 
1. 시스템 프롬프트에 "문서 근거가 없으면 모른다고 답하라"
2. Part 4의 **LLM-as-Judge** 로 검증
3. citation 강제 → 사용자가 원문 확인

### 실수 2. 검색 실패를 무시
사용자 질문이 문서 코퍼스 밖이면 **검색 결과가 0** 또는 **무관한 문서만** 옴.
이걸 그대로 프롬프트에 넣으면 모델이 **무관한 문서 기반으로 추측**.

**대응**: 검색 점수 임계치 아래면 **"정보 부족 - 담당자에게 연결"** 플로우.

### 실수 3. 문서에 잘못된 내용이 있으면 그대로 전파
RAG 는 **문서의 진실성을 검증하지 않음**.

**대응**:
1. 문서 큐레이션·버전 관리
2. 답변에 **"문서 최종 수정일"** 함께 표시
3. 주기적 평가셋 (Part 4) 으로 오답 발견·수정

### 실수 4. 모든 문제를 RAG 로 풀려고 함
"우리 회사 톤으로 답해줘" 같은 **행동 문제**는 RAG 로 안 풀림.

**대응**: RAG 는 **지식 문제** 전용. 톤·포맷은 프롬프트 + 파인튜닝 영역.

## 8. 확인 문제

1. 위 최소 예제를 내 회사 문서 (또는 임의 문서) 로 돌려 응답 차이 한 문단 정리
2. 지식 컷오프 탐침을 5가지 질문으로 실행. 어떤 질문이 "모른다"로, 어떤 질문이 "추측"으로 오는지 분류
3. 내 프로젝트의 3가지 문제를 RAG / Fine-tune 중 어디로 보낼지 정하기
4. RAG 로 "**무관한 문서**"를 일부러 주입해 모델이 어떻게 반응하는지 관찰
5. "RAG로 이걸 풀면 안 되는" 케이스를 1개 찾아 파인튜닝·프롬프트 중 뭐가 맞는지 논거 쓰기

**다음 챕터** → [Ch 10. 임베딩과 벡터 검색 기초](https://desty.github.io/study-ai-assistant-engineering/part3/10-embedding/)

"**관련 문서를 찾는**" 이 어떻게 수학적으로 되는가 — 임베딩 · 코사인 유사도 · 벡터DB.